# Cuaderno de trabajo: Pipeline de datos sobre hábitos alimenticios y peso

Este cuaderno documenta una simulación de ingeniería de datos para la actividad. El objetivo es construir un flujo completo: obtener datos, limpiarlos, organizarlos, moverlos entre sistemas, transformarlos y presentar resultados en un dashboard.

## 1. Tema y conjunto de datos elegido

El tema elegido es la relación entre hábitos alimenticios, actividad física, características corporales y niveles de peso. El conjunto de datos principal es **Estimation of Obesity Levels Based On Eating Habits and Physical Condition**, publicado en UCI Machine Learning Repository.

Según la ficha de UCI, el dataset contiene datos de personas de México, Perú y Colombia, con 17 atributos y 2.111 registros. La variable objetivo original es el nivel de obesidad, con clases como `Normal_Weight`, `Overweight_Level_I`, `Obesity_Type_I` y otras. UCI también explica que una parte fue recolectada directamente mediante plataforma web y otra fue generada sintéticamente con Weka y SMOTE.

## 2. Origen y método de obtención

La extracción se realiza automáticamente desde el archivo ZIP público de UCI. El script `scripts/run_pipeline.py` descarga el ZIP, localiza el CSV interno y lo guarda en la zona cruda `data/raw/obesity_source.csv`.

Para que el trabajo sea reproducible en computadores sin internet, el pipeline incluye una muestra local de respaldo. Esta muestra no reemplaza la fuente oficial, solo permite ejecutar la demostración cuando la descarga falla.

In [ ]:
from pathlib import Path
import json

ROOT = Path('..').resolve()
print(ROOT)

## 3. Ejecución del pipeline

La siguiente celda ejecuta el pipeline completo. En un ambiente local se puede correr directamente desde terminal con `python scripts/run_pipeline.py`.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_pipeline.py')], check=True)

## 4. Limpieza y organización

Las tareas de limpieza son:

- Convertir variables numéricas: edad, estatura, peso, actividad física, consumo de agua, uso de tecnología y número de comidas.
- Normalizar respuestas `yes/no` a valores consistentes en español: `Si`, `No` o `No informado`.
- Calcular el índice de masa corporal, IMC = peso / estatura².
- Crear una categoría de IMC calculada: bajo peso, normal, sobrepeso u obesidad.
- Crear rangos de edad para facilitar visualizaciones.
- Crear un puntaje educativo de riesgo de hábitos (`habit_risk_score`).

La limpieza convierte el dataset en una tabla más útil para análisis y dashboard.

In [ ]:
summary_path = ROOT / 'data' / 'mart' / 'pipeline_summary.json'
dashboard_path = ROOT / 'data' / 'mart' / 'dashboard_data.json'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
dashboard = json.loads(dashboard_path.read_text(encoding='utf-8'))

summary['source'], dashboard['kpis']

## 5. Movimiento de datos entre sistemas

El pipeline simula una arquitectura por capas:

1. **Raw CSV**: archivo descargado o muestra local en `data/raw/`.
2. **Staging SQLite**: base `data/staging/obesity_staging.db`, tabla `raw_obesity`, con los datos tal como llegaron.
3. **Warehouse SQLite**: base `data/warehouse/obesity_warehouse.db`, tabla `fact_obesity_habits`, con datos limpios y columnas derivadas.
4. **Data mart**: archivos `data/mart/obesity_curated.csv` y `data/mart/dashboard_data.json` para consumo analítico.

Este movimiento muestra cómo pasar de una fuente externa a una capa preparada para visualización.

In [ ]:
for name, path in summary['outputs'].items():
    print(f'{name}: {path}')

## 6. Transformaciones para formato requerido

El dashboard no necesita el dataset crudo completo, sino indicadores y agregaciones. Por eso se genera un JSON con KPIs, conteos y promedios agrupados.

Ejemplos de transformaciones:

- Conteo de personas por nivel de obesidad original.
- Conteo por categoría de IMC calculada.
- Riesgo promedio por grupo de edad.
- Distribución por medio de transporte.
- Tabla de muestra con columnas relevantes.

In [ ]:
print('Registros:', dashboard['kpis']['records'])
print('IMC promedio:', dashboard['kpis']['avg_bmi'])
print('Porcentaje sobrepeso u obesidad:', dashboard['kpis']['obesity_or_overweight_pct'])
print('Riesgo promedio:', dashboard['kpis']['avg_habit_risk_score'])

## 7. Dashboard desplegable

El dashboard se encuentra en la carpeta `dashboard/`. Para ejecutarlo:

```powershell
python dashboard/server.py
```

Luego se abre `http://127.0.0.1:8000`. El dashboard ejecuta el pipeline al iniciar y tiene un botón para actualizar los datos. Esto cumple la condición de buscar automáticamente el dataset y refrescar los resultados.

## 8. Resultados finales y aprendizaje

Los resultados finales son un dataset limpio, dos bases SQLite, un archivo JSON para visualización y un dashboard. El proceso permite aprender que la ingeniería de datos incluye más que analizar un CSV: requiere controlar la extracción, conservar una capa cruda, mover datos entre sistemas, limpiar inconsistencias, crear variables útiles y entregar información comprensible.

También se aprende la importancia de la reproducibilidad. Si la fuente externa falla, el pipeline no se rompe completamente, sino que deja evidencia de la situación y usa una muestra de respaldo para fines académicos.